# Schema Evolution: Merging Schemas in Parquet and Handling Field Changes

**Module**: 08_data_sources_and_formats  
**Notebook**: 04_schema_evolution.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Spark handles schema evolution over time: adding nullable columns, renaming columns, and using mergeSchema=True in Parquet.
- **Why does this topic exist?**: Production datasets change over time. Ingestion pipelines must accommodate evolving schemas without breaking.
- **What problem does it solve?**: Schema mismatch errors when historical data lacks newly added columns.
- **Where is it used in real-world Data Engineering?**: Long-lived data lakes with evolving business entities.

---

## 2. Schema Evolution Rules: Parquet supports adding new nullable columns seamlessly via `mergeSchema`.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Schema_Evolution_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
evo_path = "/opt/spark-data/warehouse/schema_evo_test"

# Batch 1 (Version 1 Schema: id, name)
v1 = spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "name"])
v1.write.mode("overwrite").parquet(f"{evo_path}/batch1")

# Batch 2 (Version 2 Schema: id, name, email) - Added new column!
v2 = spark.createDataFrame([(3, "Charlie", "charlie@example.com")], ["id", "name", "email"])
v2.write.mode("overwrite").parquet(f"{evo_path}/batch2")

# Read combined dataset with mergeSchema
combined = spark.read.option("mergeSchema", "true").parquet(f"{evo_path}/*")
combined.printSchema()
combined.show()

---

## Experiment: Reading Without mergeSchema vs With mergeSchema

- **1. What are we testing?**: Observe schema behavior when mergeSchema is false vs true.
- **2. What do we expect?**: Without mergeSchema, Spark infers schema from a single file and may miss new columns.

In [ ]:
print("With mergeSchema=True:")
combined.show()

**3. What happened?**: Missing column in batch 1 was automatically populated with nulls.  
**4. Why did it happen?**: mergeSchema merged StructTypes across Parquet file footers.  
**5. Practical Engineering Lesson**: Enable mergeSchema when querying datasets written under evolving schemas.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Nulls in New Column
**Problem Statement**: Filter rows where email is null in evolved schema.  
**Input / Context**: combined

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Global mergeSchema Config
**Problem Statement**: Print spark.sql.parquet.mergeSchema config.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Add Nested Field
**Problem Statement**: Simulate adding a nested struct field.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Nulls in New Column):
```python
combined.filter('email IS NULL').show()
```

#### Solution 2 (Global mergeSchema Config):
```python
print(spark.conf.get('spark.sql.parquet.mergeSchema','false'))
```

#### Solution 3 (Add Nested Field):
```python
print('Nested structs can be merged with Delta Lake / Iceberg')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **mergeSchema**: Merges disparate Parquet schemas across files.
- **Backwards Compatibility**: New columns must be nullable.

### Essential Technical Terms
- `Schema Evolution`: Adapting to changing data definitions over time.
- `mergeSchema`: Parquet read option combining file footers.

### Common Engineering Mistakes
- **Changing column data types**: Incompatible type changes (e.g. String to Int) break mergeSchema.

### Interview Questions on This Topic
- **Q: How does Spark handle missing columns in older Parquet files when mergeSchema is enabled?**
  - *A*: It populates the missing column with null values for all historical rows.

### What to Study Next
Open 05_reading_and_writing_data.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')